# Physis stage 0: manifest, split and preprocessing for GRAZPEDWRI-DX

Prepares every input the Physis pipeline needs and exports it as zip files.

What comes out:

- `manifest.csv`, one row per image: `patient_id`, `study_id`, age, exclusion flags, `fold`, and the preprocessing geometry
- `fracture_boxes.csv`, fracture boxes mapped into 384x384 space
- `images_384/`, all 20,327 images resized, padded and normalized
- `annot_sample/`, 100 normal validation-fold images for growth plate annotation
- `summary.json` and a count table per age band

Pick a CPU session, not a GPU one. This stage is I/O and never touches the GPU, so a T4x2 burns weekly quota without running any faster. Expect 15 to 25 minutes.

In [1]:
import os, re, json, glob, shutil, zipfile, hashlib, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
from sklearn.model_selection import GroupKFold

# CONFIGURATION
IMG_SIZE      = 384      # long side after resize, and the side of the square canvas
PATCH         = 16
BIT_DEPTH     = 16       # 16 = full fidelity (~3 GB), 8 = half the size (~1.5 GB)
CLIP_LO, CLIP_HI = 1.0, 99.0
N_FOLDS       = 5        # fold 0 = test, fold 1 = val, fold 2..4 = train
SEED          = 1337
N_ANNOT       = 100      # normal validation images exported for manual annotation
N_ANNOT_OVERLAP = 10     # of those, the ones all three annotators label
ZIP_SHARD_GB  = 1.4
KEEP_PNG_DIR  = False    # True keeps the output as a Kaggle Dataset instead of zips

OUT = Path('/kaggle/working/physis_data')
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'images_384').mkdir(exist_ok=True)
(OUT / 'annot_sample').mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED)
print('configuration ready')

konfigurasi siap


## Finding the input files

The Kaggle folder layout changes between dataset versions, so nothing here is hardcoded. This cell locates `dataset.csv`, the image directories and the YOLO labels itself.

In [4]:
ROOT = Path('/kaggle/input')
print('/kaggle/input holds:', [p.name for p in ROOT.iterdir()])

csv_candidates = sorted(ROOT.glob('**/dataset.csv'))
assert csv_candidates, 'dataset.csv not found'
CSV_PATH = csv_candidates[0]
BASE = CSV_PATH.parent
print('dataset.csv :', CSV_PATH)
print('base dir    :', BASE)

# image index: filestem -> path, merged across images_part1..4
img_index = {}
for p in BASE.glob('images_part*/**/*.png'):
    img_index[p.stem] = p
if not img_index:
    for p in BASE.glob('**/*.png'):
        if 'label' not in str(p).lower():
            img_index[p.stem] = p
print('images found  :', len(img_index))

# YOLO label index: filestem -> txt path
lbl_index = {}
for p in BASE.glob('**/labels/**/*.txt'):
    lbl_index[p.stem] = p
if not lbl_index:
    for p in BASE.glob('**/*.txt'):
        if p.stem in img_index:
            lbl_index[p.stem] = p
print('labels found  :', len(lbl_index))

yaml_hits = sorted(BASE.glob('**/*.yaml')) + sorted(BASE.glob('**/*.yml'))
print('yaml            :', [str(y.relative_to(BASE)) for y in yaml_hits][:5])

isi /kaggle/input: ['datasets']
dataset.csv : /kaggle/input/datasets/jasonroggy/grazpedwri-dx/dataset.csv
base dir    : /kaggle/input/datasets/jasonroggy/grazpedwri-dx
citra ditemukan : 20327
label ditemukan : 20327
yaml            : ['folder_structure/yolov5/meta.yaml']


## Loading dataset.csv and mapping its columns

Seventeen columns, and their names are not guaranteed to be what you expect. Names are normalized, matched by substring, and the matches printed so you can check them yourself.

In [5]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
print('columns:', list(df.columns))
print('rows:', len(df))
display(df.head(3))

def find_col(*keys, required=True):
    for k in keys:
        for c in df.columns:
            if k in c:
                return c
    if required:
        raise KeyError(f'no column matched {keys}; check the list printed above')
    return None

COL = {
    'stem'      : find_col('filestem', 'file_stem', 'filename'),
    'patient'   : find_col('patient_id', 'patient'),
    'age'       : find_col('age'),
    'gender'    : find_col('gender', 'sex'),
    'projection': find_col('projection', 'view', required=False),
    'laterality': find_col('laterality', 'side', required=False),
    'frac_vis'  : find_col('fracture_visible', 'fracture_v', required=False),
    'ao'        : find_col('ao_class', 'ao_'), 
    'cast'      : find_col('cast'),
    'metal_tag' : find_col('metal', required=False),
}
print()
for k, v in COL.items():
    print(f'{k:11s} -> {v}')

kolom: ['filestem', 'patient_id', 'study_number', 'timehash', 'gender', 'age', 'laterality', 'projection', 'initial_exam', 'ao_classification', 'cast', 'diagnosis_uncertain', 'osteopenia', 'fracture_visible', 'metal', 'pixel_spacing', 'device_manufacturer']
baris: 20327


,filestem,patient_id,study_number,timehash,gender,age,laterality,projection,initial_exam,ao_classification,cast,diagnosis_uncertain,osteopenia,fracture_visible,metal,pixel_spacing,device_manufacturer
0,0001_1297860395_01_WRI-L1_M014,1,1,1297860395,M,14.1,L,1,1.0,23r-M/2.1,NaN,NaN,NaN,NaN,NaN,0.144,Siemens
1,0001_1297860435_01_WRI-L2_M014,1,1,1297860435,M,14.1,L,2,1.0,23r-M/2.1,NaN,NaN,NaN,1.0,NaN,0.144,Siemens
2,0002_0354485735_01_WRI-R1_F012,2,1,354485735,F,12.0,R,1,1.0,23r-M/2.1,NaN,1.0,NaN,NaN,NaN,0.144,Siemens



stem        -> filestem
patient     -> patient_id
age         -> age
gender      -> gender
projection  -> projection
laterality  -> laterality
frac_vis    -> fracture_visible
ao          -> ao_classification
cast        -> cast
metal_tag   -> metal


## Reading the YOLO labels

Nine box classes. The index order comes from `data.yaml` when it is present, otherwise from the alphabetical order the official release uses.

Either way the guess is verified against the box counts published on the dataset card. A wrong class order swaps `fracture` for `metal` and raises no error at all, so this check is the only thing standing between that and a ruined run.

In [6]:
DEFAULT_CLASSES = ['boneanomaly','bonelesion','foreignbody','fracture','metal',
                   'periostealreaction','pronatorsign','softtissue','text']
CLASSES = DEFAULT_CLASSES
for y in yaml_hits:
    txt = Path(y).read_text(errors='ignore')
    m = re.search(r'names\s*:\s*\[(.*?)\]', txt, re.S)
    if m:
        parsed = [s.strip().strip('\'"') for s in m.group(1).split(',')]
        if len(parsed) == 9:
            CLASSES = parsed
            print('class order from', y.name, ':', CLASSES)
            break
else:
    print('using the default order:', CLASSES)

presence = {c: defaultdict(int) for c in CLASSES}
frac_boxes_raw = defaultdict(list)   # stem -> [(xc, yc, w, h) normalized]

for stem, lp in lbl_index.items():
    try:
        lines = lp.read_text().strip().splitlines()
    except Exception:
        continue
    for ln in lines:
        parts = ln.split()
        if len(parts) < 5:
            continue
        ci = int(float(parts[0]))
        if ci >= len(CLASSES):
            continue
        cname = CLASSES[ci]
        presence[cname][stem] += 1
        if cname == 'fracture':
            xc, yc, w, h = map(float, parts[1:5])
            frac_boxes_raw[stem].append((xc, yc, w, h))

counts = {c: sum(presence[c].values()) for c in CLASSES}
print('\nboxes per class:')
for c in CLASSES:
    print(f'  {c:20s} {counts[c]:>6d}')

EXPECTED = {'boneanomaly':276,'bonelesion':45,'foreignbody':8,'fracture':18090,
            'metal':818,'periostealreaction':3453,'pronatorsign':567,
            'softtissue':464,'text':23722}
mismatch = {c: (counts.get(c), EXPECTED[c]) for c in EXPECTED
            if abs(counts.get(c, 0) - EXPECTED[c]) > max(5, 0.02 * EXPECTED[c])}
if mismatch:
    print('\nWARNING, counts do not match the dataset card:', mismatch)
    print('the class order is probably different; check before going on')
else:
    print('\nbox counts match the dataset card, so the class order is right')

memakai urutan default: ['boneanomaly', 'bonelesion', 'foreignbody', 'fracture', 'metal', 'periostealreaction', 'pronatorsign', 'softtissue', 'text']

jumlah kotak per kelas:
  boneanomaly             276
  bonelesion               45
  foreignbody               8
  fracture              18090
  metal                   818
  periostealreaction     3453
  pronatorsign            567
  softtissue              464
  text                  23722

jumlah kotak cocok dengan kartu dataset, urutan kelas benar


## Deriving study_id

The filename pattern `PPPP_IIIIIIIIII_SS_...` carries the patient index, the image id and the study number. A radiologist's queue is ordered by study rather than by image, so aggregating scores needs this column. The dataset card reports 10,643 studies, and this cell checks the derived count lands near it.

In [7]:
def derive_study(stem):
    parts = stem.split('_')
    if len(parts) >= 3:
        return f'{parts[0]}_{parts[2]}'
    return parts[0]

df['stem']     = df[COL['stem']].astype(str).str.replace(r'\.png$', '', regex=True)
df['study_id'] = df['stem'].map(derive_study)
n_study = df['study_id'].nunique()
print('unique studies :', n_study, '(dataset card: 10,643)')
print('unique patients:', df[COL['patient']].nunique(), '(dataset card: 6,091)')
print('images         :', len(df))
print('images per study:', round(len(df) / n_study, 2))

studi unik   : 10699 (kartu dataset: 10.643)
pasien unik  : 6091 (kartu dataset: 6.091)
citra        : 20327
rerata citra per studi : 1.9


## Building the exclusion flags

Eight categories are dropped from the pretraining set. Two are easy to miss.

**A non-empty AO classification.** An image can carry an AO class with no fracture box, which is what happens when a fracture is known to be present but is not clearly visualized in that projection. Without this filter those occult fractures land in the clean set and the model learns that appearance as normal.

**`pronatorsign` and `softtissue`.** Both are indirect signs of injury rather than bone findings, so an image carrying either may well hold a fracture nobody boxed.

`text` is deliberately not filtered. Letter markers appear on nearly every image, so filtering on them leaves nothing behind. The model learns them as part of normal, which is one more reason the triage score uses the 95th percentile rather than the maximum.

In [8]:
def has(cls):
    return df['stem'].map(lambda s: presence[cls].get(s, 0) > 0)

df['lbl_fracture']     = has('fracture')
df['lbl_metal']        = has('metal')
df['lbl_periosteal']   = has('periostealreaction')
df['lbl_pronator']     = has('pronatorsign')
df['lbl_softtissue']   = has('softtissue')
df['lbl_boneanomaly']  = has('boneanomaly')
df['lbl_bonelesion']   = has('bonelesion')
df['lbl_foreignbody']  = has('foreignbody')
df['n_fracture_box']   = df['stem'].map(lambda s: presence['fracture'].get(s, 0))

ao = df[COL['ao']].astype(str).str.strip().str.lower()
df['tag_ao'] = ~ao.isin(['', 'nan', 'none', '0', 'na'])

def as_bool(col):
    if col is None:
        return pd.Series(False, index=df.index)
    s = df[col]
    if s.dtype == object:
        return s.astype(str).str.strip().str.lower().isin(['1','true','yes','y'])
    return s.fillna(0).astype(float) > 0

df['tag_cast']     = as_bool(COL['cast'])
df['tag_frac_vis'] = as_bool(COL['frac_vis'])

EXCL_STRICT = ['lbl_fracture','tag_frac_vis','tag_ao','lbl_metal','tag_cast',
               'lbl_periosteal','lbl_pronator','lbl_softtissue',
               'lbl_boneanomaly','lbl_bonelesion','lbl_foreignbody']
EXCL_LOOSE  = [c for c in EXCL_STRICT if c != 'tag_cast']

df['clean_strict'] = ~df[EXCL_STRICT].any(axis=1)
df['clean_loose']  = ~df[EXCL_LOOSE].any(axis=1)

print('images caught by each filter:')
for c in EXCL_STRICT:
    print(f'  {c:16s} {int(df[c].sum()):>6d}')

print()
print('clean STRICT :', int(df.clean_strict.sum()),
      'from', int(df[df.clean_strict][COL['patient']].nunique()), 'patients')
print('clean LOOSE  :', int(df.clean_loose.sum()),
      'from', int(df[df.clean_loose][COL['patient']].nunique()), 'patients')
print('cast on', round(100 * df.tag_cast.mean(), 1), '% of images')

only_ao = df.tag_ao & ~df.lbl_fracture & ~df.tag_frac_vis
print('\noccult fractures caught only by the AO filter:', int(only_ao.sum()))

kontribusi tiap filter (jumlah citra yang dikenai):
  lbl_fracture      13550
  tag_frac_vis      13550
  tag_ao            14158
  lbl_metal           707
  tag_cast           5776
  lbl_periosteal     2235
  lbl_pronator        566
  lbl_softtissue      439
  lbl_boneanomaly     192
  lbl_bonelesion       42
  lbl_foreignbody       8

citra bersih KETAT  : 5639 dari 2671 pasien
citra bersih LONGGAR: 5672 dari 2682 pasien
gips pada 28.4 % citra

fraktur okulta yang tertangkap khusus oleh filter AO: 773


## Splitting by patient

GroupKFold on `patient_id`, run once here and reused by every experiment. At 3.3 images per patient, a random per-image split puts the same child in train and test.

Fold 0 becomes test, fold 1 validation, the rest train. Normalization statistics and every threshold touch the validation fold only.

The assignment is deterministic rather than seeded. GroupKFold does not shuffle unless asked, so `SEED` never reaches it and the folds reproduce from the patient ids alone.

In [9]:
gkf = GroupKFold(n_splits=N_FOLDS)
df['fold'] = -1
for i, (_, idx) in enumerate(gkf.split(df, groups=df[COL['patient']])):
    df.loc[df.index[idx], 'fold'] = i

df['split'] = np.select(
    [df.fold == 0, df.fold == 1],
    ['test', 'val'],
    default='train')

chk = df.groupby('split').agg(
    images=('stem', 'size'),
    patients=(COL['patient'], 'nunique'),
    studies=('study_id', 'nunique'),
    clean_strict=('clean_strict', 'sum'),
    fractures=('lbl_fracture', 'sum'))
display(chk)

overlap = (df.groupby(COL['patient'])['split'].nunique() > 1).sum()
print('patients appearing in more than one split:', overlap, '(must be 0)')
assert overlap == 0

,citra,pasien,studi,bersih_ketat,fraktur
split,,,,,
test,4066,1218,2139,1157,2667
train,12195,3654,6395,3382,8141
val,4066,1219,2165,1100,2742


pasien yang muncul di lebih dari satu split: 0 (harus 0)


## Age distribution per band

Normalization statistics are computed per age band, so a thin band gives an unstable estimate. This cell flags any band holding fewer than 50 clean validation images. The bottom of the range is almost certainly thin, and that is worth knowing now rather than while writing up results.

In [10]:
df['age'] = pd.to_numeric(df[COL['age']], errors='coerce')
print('age range  :', round(df.age.min(), 2), 'to', round(df.age.max(), 2))
print('missing age:', int(df.age.isna().sum()))

df['age_band'] = np.floor(df.age).astype('Int64')
band = (df[df.clean_strict]
        .groupby(['age_band', 'split'])
        .size().unstack(fill_value=0)
        .reindex(columns=['train','val','test'], fill_value=0))
band['total'] = band.sum(axis=1)
display(band)

thin = band[band['val'] < 50].index.tolist()
print('bands with fewer than 50 clean validation images:', thin)
print('these need merging or widening before normalization statistics')

rentang usia: 0.2 sampai 19.0
usia hilang : 0


split,train,val,test,total
age_band,,,,
0,9,0,0,9
1,46,12,12,70
2,53,16,20,89
3,53,10,17,80
4,62,23,10,95
5,37,25,22,84
6,61,10,21,92
7,93,37,27,157
8,156,48,55,259


band dengan < 50 citra bersih di fold val: [0, 1, 2, 3, 4, 5, 6, 7, 8, 18, 19]
band ini perlu digabung atau dilebarkan sebelum menghitung (mu_a, sigma_a)


## Preprocessing

For each image: resize the long side to 384 preserving aspect ratio, clip intensity at the 1st and 99th percentile, scale to [0, 1], then zero-pad symmetrically to a 384x384 canvas.

The percentiles are computed before padding. Taking them afterwards lets the added zeros, about 43% of the canvas, drag the 1st percentile to zero and flatten the contrast of every image by an amount that depends on its aspect ratio.

Padding offsets go into the manifest because two things need them: mapping fracture boxes into 384 space, and masking padding patches during training.

In [11]:
from multiprocessing import Pool, cpu_count

MAXV = 65535 if BIT_DEPTH == 16 else 255
DTYPE = np.uint16 if BIT_DEPTH == 16 else np.uint8

def preprocess_one(args):
    stem, src = args
    try:
        im = cv2.imread(str(src), cv2.IMREAD_UNCHANGED)
        if im is None:
            return (stem, None)
        if im.ndim == 3:
            im = im[..., 0]
        h0, w0 = im.shape
        scale = IMG_SIZE / max(h0, w0)
        nw, nh = max(1, int(round(w0 * scale))), max(1, int(round(h0 * scale)))
        im = cv2.resize(im, (nw, nh), interpolation=cv2.INTER_AREA).astype(np.float32)

        lo, hi = np.percentile(im, [CLIP_LO, CLIP_HI])
        if hi <= lo:
            lo, hi = float(im.min()), float(max(im.max(), im.min() + 1))
        im = np.clip((im - lo) / (hi - lo), 0, 1)

        canvas = np.zeros((IMG_SIZE, IMG_SIZE), np.float32)
        px, py = (IMG_SIZE - nw) // 2, (IMG_SIZE - nh) // 2
        canvas[py:py+nh, px:px+nw] = im
        cv2.imwrite(str(OUT / 'images_384' / f'{stem}.png'),
                    (canvas * MAXV).astype(DTYPE))
        return (stem, dict(orig_w=w0, orig_h=h0, scale=scale,
                           new_w=nw, new_h=nh, pad_x=px, pad_y=py,
                           clip_lo=float(lo), clip_hi=float(hi)))
    except Exception as e:
        return (stem, None)

jobs = [(s, img_index[s]) for s in df['stem'] if s in img_index]
print('processing', len(jobs), 'images across', cpu_count(), 'processes')

geo = {}
with Pool(cpu_count()) as pool:
    for i, (stem, info) in enumerate(pool.imap_unordered(preprocess_one, jobs, chunksize=64)):
        if info is not None:
            geo[stem] = info
        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(jobs)}')

print('done:', len(geo), 'failed:', len(jobs) - len(geo))
for k in ['scale','new_w','new_h','pad_x','pad_y']:
    df[k] = df['stem'].map(lambda s: geo.get(s, {}).get(k, np.nan))
df['preprocessed'] = df['stem'].isin(geo)

memproses 20327 citra dengan 4 proses
  2000/20327
  4000/20327
  6000/20327
  8000/20327
  10000/20327
  12000/20327
  14000/20327
  16000/20327
  18000/20327
  20000/20327
berhasil: 20327 gagal: 0


## Mapping fracture boxes into 384 space

YOLO labels use coordinates normalized against the original image, and resizing and padding move them. This cell rewrites each box in 384-space pixels and in patch indices, so the 50% coverage rule for patch labels can be applied directly with no further transform.

In [12]:
rows = []
for stem, boxes in frac_boxes_raw.items():
    g = geo.get(stem)
    if g is None:
        continue
    for (xc, yc, bw, bh) in boxes:
        cx = xc * g['new_w'] + g['pad_x']
        cy = yc * g['new_h'] + g['pad_y']
        w  = bw * g['new_w']
        h  = bh * g['new_h']
        x0, y0 = max(0.0, cx - w/2), max(0.0, cy - h/2)
        x1, y1 = min(float(IMG_SIZE), cx + w/2), min(float(IMG_SIZE), cy + h/2)
        rows.append(dict(stem=stem, x0=x0, y0=y0, x1=x1, y1=y1,
                         patch_i0=int(x0 // PATCH), patch_j0=int(y0 // PATCH),
                         patch_i1=int((x1 - 1e-6) // PATCH),
                         patch_j1=int((y1 - 1e-6) // PATCH)))

fb = pd.DataFrame(rows)
fb.to_csv(OUT / 'fracture_boxes.csv', index=False)
print('fracture boxes exported:', len(fb), 'across', fb.stem.nunique(), 'images')
display(fb.head())

area = ((fb.x1 - fb.x0) * (fb.y1 - fb.y0)) / (PATCH * PATCH)
print('\nbox area in 16x16 patches:')
print(area.describe().round(1))

kotak fraktur diekspor: 18090 pada 13550 citra


,stem,x0,y0,x1,y1,patch_i0,patch_j0,patch_i1,patch_j1
0,5533_1055512321_02_WRI-L1_F008,189.074145,268.548864,236.721315,324.182784,11,16,14,20
1,5533_1055512321_02_WRI-L1_F008,143.935005,250.562112,173.191935,275.659968,8,15,10,17
2,2007_0850762755_05_WRI-R2_F008,159.053082,198.309888,215.469078,234.816768,9,12,13,14
3,0313_0947475228_06_WRI-R2_F015,176.518490,189.061248,240.074070,238.694016,11,11,15,14
4,3863_1014196661_02_WRI-L1_M015,176.901020,263.746560,230.300340,299.896320,11,16,14,18



luas kotak dalam satuan patch 16x16:
count    18090.0
mean         6.2
std          4.2
min          0.5
25%          3.4
50%          5.3
75%          7.9
max         61.6
dtype: float64


## Sample for growth plate annotation

One hundred normal images, drawn only from validation-fold patients so annotation never touches test. Ten are marked for all three team members to label, which is what inter-annotator agreement is computed from.

Exported as 8-bit PNG because they are only for human eyes, and already in 384 space so the coordinates can serve as patch boxes without conversion.

In [13]:
pool_val = df[(df.split == 'val') & df.clean_strict & df.preprocessed].copy()
pool_val = pool_val.dropna(subset=['age'])

# stratify by age band so the sample is not drawn from one age group
picked = (pool_val.sample(frac=1.0, random_state=SEED)
          .groupby('age_band', group_keys=False)
          .head(max(1, N_ANNOT // max(1, pool_val.age_band.nunique()))))
if len(picked) < N_ANNOT:
    extra = pool_val[~pool_val.stem.isin(picked.stem)].sample(
        N_ANNOT - len(picked), random_state=SEED)
    picked = pd.concat([picked, extra])
picked = picked.head(N_ANNOT).reset_index(drop=True)
picked['overlap'] = picked.index < N_ANNOT_OVERLAP

for s in picked.stem:
    im = cv2.imread(str(OUT / 'images_384' / f'{s}.png'), cv2.IMREAD_UNCHANGED)
    if BIT_DEPTH == 16:
        im = (im / 257).astype(np.uint8)
    cv2.imwrite(str(OUT / 'annot_sample' / f'{s}.png'), im)

picked[['stem', COL['patient'], 'study_id', 'age', 'overlap']].to_csv(
    OUT / 'annot_sample' / 'annot_list.csv', index=False)

template = pd.DataFrame(columns=['stem','structure','x0','y0','x1','y1','status','annotator'])
template.to_csv(OUT / 'annot_sample' / 'physis_boxes_TEMPLATE.csv', index=False)

print('annotation sample:', len(picked), '| three-annotator overlap:', int(picked.overlap.sum()))
print('age band spread:'); print(picked.age_band.value_counts().sort_index())

sampel anotasi: 100 | overlap tiga anotator: 10
sebaran band usia:
age_band
1     5
2     5
3     5
4     5
5     6
6     5
7     5
8     5
9     5
10    6
11    5
12    8
13    6
14    7
15    5
16    6
17    7
18    4
Name: count, dtype: Int64


## Writing the manifest and summary

In [14]:
keep = ['stem', COL['patient'], 'study_id', 'age', 'age_band', COL['gender']]
for c in [COL['projection'], COL['laterality']]:
    if c: keep.append(c)
keep += ['fold','split','clean_strict','clean_loose','n_fracture_box',
         'lbl_fracture','lbl_metal','lbl_periosteal','lbl_pronator','lbl_softtissue',
         'lbl_boneanomaly','lbl_bonelesion','lbl_foreignbody',
         'tag_ao','tag_cast','tag_frac_vis',
         'orig_ok','scale','new_w','new_h','pad_x','pad_y','preprocessed']
df['orig_ok'] = df['preprocessed']
manifest = df[[c for c in keep if c in df.columns]].rename(
    columns={COL['patient']: 'patient_id', COL['gender']: 'gender'})
manifest.to_csv(OUT / 'manifest.csv', index=False)

summary = dict(
    n_images=int(len(df)), n_patients=int(df[COL['patient']].nunique()),
    n_studies=int(df.study_id.nunique()),
    n_clean_strict=int(df.clean_strict.sum()), n_clean_loose=int(df.clean_loose.sum()),
    n_clean_strict_train=int(df[(df.split=='train') & df.clean_strict].shape[0]),
    n_clean_strict_val=int(df[(df.split=='val') & df.clean_strict].shape[0]),
    cast_rate=float(df.tag_cast.mean()),
    occult_caught_by_ao=int(only_ao.sum()),
    img_size=IMG_SIZE, patch=PATCH, bit_depth=BIT_DEPTH,
    clip=[CLIP_LO, CLIP_HI], n_folds=N_FOLDS, seed=SEED,
    class_order=CLASSES, thin_val_bands=[int(b) for b in thin],
    preprocess_failed=int((~df.preprocessed).sum()))
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2))
band.to_csv(OUT / 'age_band_counts.csv')

print(json.dumps(summary, indent=2))

{
  "n_images": 20327,
  "n_patients": 6091,
  "n_studies": 10699,
  "n_clean_strict": 5639,
  "n_clean_loose": 5672,
  "n_clean_strict_train": 3382,
  "n_clean_strict_val": 1100,
  "cast_rate": 0.28415408077925913,
  "occult_caught_by_ao": 773,
  "img_size": 384,
  "patch": 16,
  "bit_depth": 16,
  "clip": [
    1.0,
    99.0
  ],
  "n_folds": 5,
  "seed": 1337,
  "class_order": [
    "boneanomaly",
    "bonelesion",
    "foreignbody",
    "fracture",
    "metal",
    "periostealreaction",
    "pronatorsign",
    "softtissue",
    "text"
  ],
  "thin_val_bands": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    18,
    19
  ],
  "preprocess_failed": 0
}


In [15]:
readme = f'''# Physis, ready-to-use data

Produced by the stage 0 notebook from GRAZPEDWRI-DX (Kaggle: jasonroggy/grazpedwri-dx, CC0).

## Contents
- manifest.csv           one row per image; `split` and `fold` are final
- fracture_boxes.csv     fracture boxes in {IMG_SIZE}x{IMG_SIZE} space, plus patch indices
- age_band_counts.csv    clean images per age band per split
- summary.json           every parameter and headline number
- images_384/            {len(geo)} images, {IMG_SIZE}x{IMG_SIZE}, {BIT_DEPTH}-bit, padded
- annot_sample/          {len(picked)} normal validation images for growth plate annotation

## Preprocessing
resize the long side to {IMG_SIZE}, preserving aspect ratio
-> clip at the {CLIP_LO}/{CLIP_HI} percentile of the image before padding
-> scale to [0,1] -> zero-pad symmetrically to {IMG_SIZE}x{IMG_SIZE}

Patches that are entirely padding MUST be excluded from context sampling, target
sampling, and any surprise map. Use pad_x, pad_y, new_w, new_h from the manifest:
patch (i,j) is valid when 16*i >= pad_x, 16*(i+1) <= pad_x+new_w, and likewise for j.

## Split rule
GroupKFold over {N_FOLDS} folds on patient_id, deterministic rather than seeded:
GroupKFold does not shuffle unless asked, so the folds follow from the patient ids
alone. fold 0 = test, fold 1 = val, folds 2..4 = train. No patient crosses a split.

## Clean image definition
clean_strict drops: fracture, fracture_visible, a non-empty AO classification,
metal, cast, periostealreaction, pronatorsign, softtissue, boneanomaly,
bonelesion, foreignbody.
clean_loose is the same but keeps cast.
`text` is deliberately not filtered, because it appears on nearly every image.
'''
(OUT / 'README.md').write_text(readme)
print(readme)

# Physis — data siap pakai

Dihasilkan oleh notebook Tahap 0 dari GRAZPEDWRI-DX (Kaggle: jasonroggy/grazpedwri-dx, CC0).

## Isi
- manifest.csv           satu baris per citra; kolom `split` dan `fold` sudah final
- fracture_boxes.csv     kotak fraktur di ruang 384x384 dan indeks patch
- age_band_counts.csv    jumlah citra bersih per band usia per split
- summary.json           seluruh parameter dan angka ringkas
- images_384/            20327 citra, 384x384, 16-bit, sudah dipad
- annot_sample/          100 citra normal fold val untuk anotasi lempeng pertumbuhan

## Prapemrosesan
resize sisi panjang ke 384 (rasio aspek dipertahankan)
-> potong persentil 1.0/99.0 pada citra sebelum padding
-> skala ke [0,1] -> pad nol simetris ke 384x384

Patch yang seluruhnya padding HARUS dikeluarkan dari sampling context, sampling
target, dan peta surprise. Gunakan pad_x, pad_y, new_w, new_h dari manifest:
patch (i,j) valid bila 16*i >= pad_x, 16*(i+1) <= pad_x+new_w, dan setara untuk j.

## Aturan sp

## Zipping

Metadata is separated from the images so the small files can be downloaded first. Images are split into shards because a single large download from Kaggle tends to break partway.

With `KEEP_PNG_DIR = True` the PNG directory survives, and the notebook output can be saved as a Kaggle Dataset and attached to another notebook with nothing to download.

In [16]:
ZIPDIR = Path('/kaggle/working/zips'); ZIPDIR.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIPDIR / 'physis_meta.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['manifest.csv','fracture_boxes.csv','age_band_counts.csv',
              'summary.json','README.md']:
        z.write(OUT / f, f)
    for f in sorted((OUT / 'annot_sample').iterdir()):
        z.write(f, f'annot_sample/{f.name}')
print('physis_meta.zip :', round((ZIPDIR/'physis_meta.zip').stat().st_size/1e6, 1), 'MB')

pngs = sorted((OUT / 'images_384').iterdir())
limit = ZIP_SHARD_GB * 1e9
shard, cur, size = 1, [], 0
def flush(shard, files):
    if not files: return
    p = ZIPDIR / f'physis_images_{shard:02d}.zip'
    with zipfile.ZipFile(p, 'w', zipfile.ZIP_STORED) as z:
        for f in files:
            z.write(f, f'images_384/{f.name}')
    print(p.name, ':', round(p.stat().st_size/1e9, 2), 'GB', f'({len(files)} files)')

for f in pngs:
    s = f.stat().st_size
    if size + s > limit and cur:
        flush(shard, cur); shard += 1; cur, size = [], 0
    cur.append(f); size += s
flush(shard, cur)

if not KEEP_PNG_DIR:
    shutil.rmtree(OUT / 'images_384')
    print('\nPNG directory removed, only the zips remain')
print('\nready to download from /kaggle/working/zips')

physis_meta.zip : 6.1 MB
physis_images_01.zip : 1.4 GB (9904 berkas)
physis_images_02.zip : 1.4 GB (9926 berkas)
physis_images_03.zip : 0.07 GB (497 berkas)

direktori PNG dihapus, hanya zip yang tersisa

siap diunduh dari /kaggle/working/zips


## What to do next

Download `physis_meta.zip` first and read `summary.json`. `n_clean_strict` decides whether stage A can train from scratch or has to start from ImageNet, and whether the experiment plan fits the compute budget. Well under 3,000 means falling back to `clean_loose`, and that cast decision belongs in the limitations.

Check `thin_val_bands`. Any band listed there needs merging before normalization statistics are computed.

Download the image shards, or save this notebook's output as a Kaggle Dataset and pull it through the Kaggle API on the training machine.

Hand `annot_sample/` to the team for growth plate annotation, filling in `physis_boxes_TEMPLATE.csv`.

Commit `manifest.csv`, `fracture_boxes.csv`, `summary.json` and this notebook. The split is final from here, and every experiment reads it from the same files.